<a href="https://colab.research.google.com/github/Dechenlama0505/BudgetBot-AI-Dataset/blob/main/BudgetBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

df = pd.read_csv("budget_data.csv")
print(df.head())

                        date    category  amount
0  2022-07-06 05:57:10 +0000  Restuarant    5.50
1  2022-07-06 05:57:27 +0000      Market    2.00
2  2022-07-06 05:58:12 +0000       Coffe   30.10
3  2022-07-06 05:58:25 +0000      Market   17.33
4  2022-07-06 05:59:00 +0000  Restuarant    5.50


In [2]:
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day

In [3]:
df['category'] = df['category'].str.strip().str.lower()

In [4]:
grouped = df.groupby(['year', 'month', 'category'])

In [5]:
training_data = []

for (year, month, category), group in grouped:
    group = group.sort_values('date')

    total_final = group['amount'].sum()

    cumulative_spent = 0

    for i, row in group.iterrows():
        cumulative_spent += row['amount']

        day_of_month = row['day']
        transactions = group[group['day'] <= day_of_month].shape[0]

        avg_daily_spend = cumulative_spent / day_of_month

        training_data.append({
            'day_of_month': day_of_month,
            'category': category,
            'spent_so_far': cumulative_spent,
            'transactions': transactions,
            'avg_daily_spend': avg_daily_spend,
            'final_spend': total_final
        })

In [6]:
train_df = pd.DataFrame(training_data)
print(train_df.head())

   day_of_month  category  spent_so_far  transactions  avg_daily_spend  \
0            28  clothing          87.0             3         3.107143   
1            28  clothing         222.5             3         7.946429   
2            28  clothing         312.5             3        11.160714   
3             6     coffe          30.1             3         5.016667   
4             6     coffe          36.7             3         6.116667   

   final_spend  
0        312.5  
1        312.5  
2        312.5  
3        207.1  
4        207.1  


In [7]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
train_df['category'] = le.fit_transform(train_df['category'])

In [8]:
X = train_df[['day_of_month', 'category', 'spent_so_far', 'transactions', 'avg_daily_spend']]
y = train_df['final_spend']

In [9]:
# Random Forest

In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import numpy as np

# Features and target
X = train_df[['day_of_month', 'category', 'spent_so_far', 'transactions', 'avg_daily_spend']]
y = train_df['final_spend']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train model
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

In [13]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

MAE: 43.3451392319615
RMSE: 82.47644156554291
R2 Score: 0.8382520463267291


In [14]:
joblib.dump(model, "budget_overrun_model.pkl")
joblib.dump(le, "category_encoder.pkl")

['category_encoder.pkl']

In [15]:
print(le.classes_)

['business lunch' 'business_expenses' 'clothing' 'coffe' 'communal'
 'events' 'film/enjoyment' 'fuel' 'health' 'joy' 'learning' 'market'
 'motel' 'other' 'phone' 'rent car' 'restuarant' 'sport' 'taxi' 'tech'
 'transport' 'travel']


In [17]:
#Mapping the categories to match the categories on budgetbot

In [18]:
df['category'] = df['category'].astype(str).str.strip().str.lower()

category_map = {
    'business lunch': 'Food & Drinks',
    'coffe': 'Food & Drinks',
    'restuarant': 'Food & Drinks',
    'fuel': 'Transport',
    'taxi': 'Transport',
    'rent car': 'Transport',
    'transport': 'Transport',
    'film/enjoyment': 'Entertainment',
    'events': 'Entertainment',
    'joy': 'Entertainment',
    'market': 'Groceries',
    'health': 'Health',
    'sport': 'Health',
    'learning': 'Education',
    'tech': 'Shopping',
    'clothing': 'Shopping',
    'motel': 'Travel',
    'travel': 'Travel',
    'phone': 'Mobile Bill',
    'business_expenses': 'Others',
    'other': 'Others',
    'communal': 'Home Bills'
}

df['category'] = df['category'].replace(category_map)

In [19]:
#Rebuild the training dataset after mapping

In [20]:
training_data = []

grouped = df.groupby(['year', 'month', 'category'])

for (year, month, category), group in grouped:
    group = group.sort_values('date')

    total_final = group['amount'].sum()
    cumulative_spent = 0

    for _, row in group.iterrows():
        cumulative_spent += row['amount']
        day_of_month = row['day']
        transactions = group[group['day'] <= day_of_month].shape[0]
        avg_daily_spend = cumulative_spent / day_of_month

        training_data.append({
            'day_of_month': day_of_month,
            'category': category,
            'spent_so_far': cumulative_spent,
            'transactions': transactions,
            'avg_daily_spend': avg_daily_spend,
            'final_spend': total_final
        })

train_df = pd.DataFrame(training_data)

print(train_df.head())
print(train_df['category'].unique())

   day_of_month       category  spent_so_far  transactions  avg_daily_spend  \
0             6  Food & Drinks           5.5             8         0.916667   
1             6  Food & Drinks          35.6             8         5.933333   
2             6  Food & Drinks          41.1             8         6.850000   
3             6  Food & Drinks          51.1             8         8.516667   
4             6  Food & Drinks          61.1             8        10.183333   

   final_spend  
0        377.1  
1        377.1  
2        377.1  
3        377.1  
4        377.1  
['Food & Drinks' 'Groceries' 'Home Bills' 'Mobile Bill' 'Others'
 'Shopping' 'Transport' 'Travel' 'Education' 'Entertainment' 'Health']


In [21]:
#train encoder again

In [22]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
train_df['category'] = le.fit_transform(train_df['category'])

print(le.classes_)

['Education' 'Entertainment' 'Food & Drinks' 'Groceries' 'Health'
 'Home Bills' 'Mobile Bill' 'Others' 'Shopping' 'Transport' 'Travel']


In [23]:
#Recreate X and y

In [24]:
X = train_df[['day_of_month', 'category', 'spent_so_far', 'transactions', 'avg_daily_spend']]
y = train_df['final_spend']

In [25]:
#retrainging the model due to changes in the categories

In [26]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [27]:
#testing

In [28]:
sample_food = pd.DataFrame([{
    'day_of_month': 12,
    'category': le.transform(['Food & Drinks'])[0],
    'spent_so_far': 4200,
    'transactions': 9,
    'avg_daily_spend': 350
}])

prediction_food = model.predict(sample_food)
print("Food & Drinks prediction:", prediction_food[0])

Food & Drinks prediction: 2509.2760000000017


In [29]:
sample_ent = pd.DataFrame([{
    'day_of_month': 18,
    'category': le.transform(['Entertainment'])[0],
    'spent_so_far': 3100,
    'transactions': 8,
    'avg_daily_spend': 172.22
}])

prediction_ent = model.predict(sample_ent)
print("Entertainment prediction:", prediction_ent[0])

Entertainment prediction: 2534.8820000000014


In [30]:
print(le.classes_)

['Education' 'Entertainment' 'Food & Drinks' 'Groceries' 'Health'
 'Home Bills' 'Mobile Bill' 'Others' 'Shopping' 'Transport' 'Travel']


In [31]:
import joblib

joblib.dump(model, "budget_overrun_model.pkl")
joblib.dump(le, "category_encoder.pkl")

['category_encoder.pkl']

In [32]:
print(le.classes_)

['Education' 'Entertainment' 'Food & Drinks' 'Groceries' 'Health'
 'Home Bills' 'Mobile Bill' 'Others' 'Shopping' 'Transport' 'Travel']
